In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

In [2]:
def make_cv_folds(train_pool, dates, n_splits=5):
    for fold, (train_idx, val_idx) in enumerate(TimeSeriesSplit(n_splits=n_splits).split(dates)):
        train_dates = dates[train_idx]# gives the training dates for that specific fold like what the actual dates are
        val_dates = dates[val_idx]# gives the validation dates for that specific fold like what the actual validation dates are
        assert not set(train_dates) & set(val_dates)
        train_mask = train_pool['FlightDate'].isin(train_dates)# finds the rows that correspond to the dates and returns a boolean array
        val_mask = train_pool['FlightDate'].isin(val_dates)# finds the rows that correspond to the dates and returns a boolean array
        train_fold = train_pool[train_mask]#gets the actual flights rows that belong to the training dates
        val_fold = train_pool[val_mask]# gets the actual flight rows that belong to the validation dates
        yield fold, train_fold, val_fold, train_dates, val_dates

In [3]:
df = pd.read_csv('../data/interim/seattle_ontime_clean.csv') #Loading the csv
df.shape
df.columns


Index(['Unnamed: 0', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek',
       'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline',
       'IATA_CODE_Reporting_Airline',
       ...
       'Div4TailNum', 'Div5Airport', 'Div5AirportID', 'Div5AirportSeqID',
       'Div5WheelsOn', 'Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff',
       'Div5TailNum', 'Unnamed: 109'],
      dtype='object', length=111)

In [4]:

df['FlightDate'] = pd.to_datetime(df["FlightDate"])
df['FlightDate'].head()

0   2024-08-01
1   2024-08-02
2   2024-08-03
3   2024-08-04
4   2024-08-01
Name: FlightDate, dtype: datetime64[ns]

In [5]:
post_flight = [
    'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups',
    'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'ArrTime', 'ArrDelayMinutes',
    'ArrDel15', 'ArrivalDelayGroups', 'ActualElapsedTime', 'AirTime',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
    'FirstDepTime', 'TotalAddGTime', 'LongestAddGTime',
    'Cancelled', 'CancellationCode', 'Diverted',
    'DivAirportLandings', 'DivReachedDest', 'DivActualElapsedTime', 'DivArrDelay',
    'DivDistance',
    'Div1Airport', 'Div1AirportID', 'Div1AirportSeqID', 'Div1WheelsOn',
    'Div1TotalGTime', 'Div1LongestGTime', 'Div1WheelsOff', 'Div1TailNum',
    'Div2Airport', 'Div2AirportID', 'Div2AirportSeqID', 'Div2WheelsOn',
    'Div2TotalGTime', 'Div2LongestGTime', 'Div2WheelsOff', 'Div2TailNum',
    'Div3Airport', 'Div3AirportID', 'Div3AirportSeqID', 'Div3WheelsOn',
    'Div3TotalGTime', 'Div3LongestGTime', 'Div3WheelsOff', 'Div3TailNum',
    'Div4Airport', 'Div4AirportID', 'Div4AirportSeqID', 'Div4WheelsOn',
    'Div4TotalGTime', 'Div4LongestGTime', 'Div4WheelsOff', 'Div4TailNum',
    'Div5Airport', 'Div5AirportID', 'Div5AirportSeqID', 'Div5WheelsOn',
    'Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff', 'Div5TailNum',
]
drop_cols = [
    'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'Tail_Number',
    'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID',
    'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac',
    'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID',
    'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac',
    'DepTimeBlk', 'ArrTimeBlk', 'Flights', 'DistanceGroup','Origin','Unnamed: 0', 'Unnamed: 109'
]


df = df.drop(columns=post_flight + drop_cols)
df.columns

Index(['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate',
       'IATA_CODE_Reporting_Airline', 'Flight_Number_Reporting_Airline',
       'Dest', 'CRSDepTime', 'CRSArrTime', 'ArrDelay', 'CRSElapsedTime',
       'Distance'],
      dtype='object')

## Why post-flight and redundant columns were dropped

Post-flight features like `DepDelay`, `DepTime`, `TaxiOut`, and `ActualElapsedTime` were dropped because they don't exist when someone is booking a flight. If we trained the model on these, it would pick up on things like "flights that depart late tend to arrive late" and look really accurate but that's useless because we wouldn't know the actual departure time when a user is trying to decide between two flights. We'd be feeding the model information it would never have in the real scenario and therefore it's predictions will be wrong.

The columns in `drop_cols` like `Reporting_Airline`, `OriginCityName`, `DistanceGroup`, and `DepTimeBlk` are just duplicates of information we already have in other columns. `Origin` was also dropped since every single row is SEA and therefore there's not really a pattern for the model to uncover with regards to predicting ArrDelay as both a delayed flight and an ontime flight have origin as SEA.

In [6]:
cutoff = pd.Timestamp('2025-10-01')#this is the cutoff because it correlates exactly with Q4 and also because it leaves plenty of data for training like about 88% for training and 12% for testing so that theres enough data to train on
train_pool = df[df['FlightDate'] < cutoff] #gets all the rows to train on that are before this particular date and it also happens to be about 88% of the data 
test = df[df['FlightDate'] >= cutoff] # gets all the rows to test on that are after or on this date which also happens to be about 12% of the data

dates = np.sort(train_pool['FlightDate'].unique())# sorts the dates from training pool to unique dates basically gives you all the unique dates
for fold, train_fold, val_fold, train_dates, val_dates in make_cv_folds(train_pool, dates):
    print(fold, train_fold.shape[0], val_fold.shape[0], train_dates.min(), train_dates.max(), val_dates.min(), val_dates.max())


print(len(train_pool), train_pool['FlightDate'].nunique(), train_pool['FlightDate'].min(), train_pool['FlightDate'].max())
print(len(test), test['FlightDate'].nunique(), test['FlightDate'].min(), test['FlightDate'].max())
assert train_pool['FlightDate'].max() < test['FlightDate'].min()
assert len(train_pool) + len(test) == len(df)


0 41901 51554 2024-01-01T00:00:00.000000000 2024-04-18T00:00:00.000000000 2024-04-19T00:00:00.000000000 2024-08-02T00:00:00.000000000
1 93455 49684 2024-01-01T00:00:00.000000000 2024-08-02T00:00:00.000000000 2024-08-03T00:00:00.000000000 2024-11-16T00:00:00.000000000
2 143139 41431 2024-01-01T00:00:00.000000000 2024-11-16T00:00:00.000000000 2024-11-17T00:00:00.000000000 2025-03-02T00:00:00.000000000
3 184570 47184 2024-01-01T00:00:00.000000000 2025-03-02T00:00:00.000000000 2025-03-03T00:00:00.000000000 2025-06-16T00:00:00.000000000
4 231754 53895 2024-01-01T00:00:00.000000000 2025-06-16T00:00:00.000000000 2025-06-17T00:00:00.000000000 2025-09-30T00:00:00.000000000
285649 639 2024-01-01 00:00:00 2025-09-30 00:00:00
38841 92 2025-10-01 00:00:00 2025-12-31 00:00:00


## Train pool / test split and CV strategy (Issue 2)

Single cutoff date: `2025-10-01`. Everything before it is the **train pool**
(2024-01-01 to 2025-09-30, 639 dates, 285,649 rows); everything on or after it is
**test** (2025-10-01 to 2025-12-31, 92 dates, 38,841 rows, ~12% of the data).
Test is held out untouched until Issue 10 — it is never used to compare or select
models.

There is no separate fixed validation split. Every candidate model in Issues 3, 4,
5, 8, 9 is scored across the same 5 `TimeSeriesSplit` folds over the train pool, so
every comparison uses identical folds.

`TimeSeriesSplit` is run over the array of the train pool's *unique dates*, not over
rows, because flight rows are not equally spaced (242-555 flights/day) while calendar
dates are. Each fold's row membership is recovered afterward with
`train_pool['FlightDate'].isin(fold_dates)`.

**`gap=0`** (the default) is a deliberate choice, not an oversight: no feature in this
project is lagged or rolling, every predictor is known at booking time, so adjacent
dates share no constructed value that a zero gap could leak across a fold boundary.
Revisit this if a lagged or rolling feature is ever added.

**Seasonal limitation:** because the split is temporal, the test partition falls
entirely in Q4 (Oct-Dec). The Issue 10 holdout score therefore reflects winter
operations only, not a full-year average. `Month` and `DayOfWeek` stay in the feature
set for this reason, and this caveat should be repeated at Issues 6 and 15.


In [7]:
for fold, (train_idx, val_idx) in enumerate(TimeSeriesSplit(n_splits=5).split(dates)):
    train_dates = dates[train_idx]
    val_dates = dates[val_idx]
    train_mask = train_pool['FlightDate'].isin(train_dates)
    val_mask = train_pool['FlightDate'].isin(val_dates)
    train_fold = train_pool[train_mask]
    val_fold = train_pool[val_mask]